# 6.10 — Second-Order Methods & K-FAC

Second-order optimization asks not only "which way is downhill?" but also "how curved is the hill in each direction?" K-FAC (Kronecker-Factored Approximate Curvature) makes that idea practical for neural-network layers: instead of inverting one enormous curvature matrix, it approximates the layer curvature with two small covariance factors and applies the update \(\Delta W \approx -\eta A^{-1} G S^{-1}\).

## 📖 Concept walkthrough — build each idea from scratch

Before the compact worked examples, we build second-order optimization and K-FAC one idea at a time. Run each cell in order and inspect the numbers: the goal is to see why raw gradients can be poorly scaled, how curvature rescales them, and why Kronecker structure turns an impossible inverse into two small ones. The walkthrough uses `_w` variables so it does not collide with later examples.

In [ ]:
import numpy as np  # arrays and linear algebra for every scratch calculation.
import matplotlib.pyplot as plt  # visual debugging for losses, steps, and matrices.
np.random.seed(0)  # reproducibility for minibatches and toy factors.

### 1. Raw gradient descent can be fooled by curvature

Start with the smallest possible optimization problem: a quadratic bowl \(L(w)=\tfrac12 h(w-w^*)^2\). Its gradient is \(h(w-w^*)\), so the same distance from optimum creates a much larger gradient when curvature \(h\) is large. A single global learning rate must be tiny enough for the steep direction, which makes shallow directions crawl.

In [ ]:
w_grid_w = np.linspace(-1.0, 4.0, 120)  # possible parameter values to visualize.
w_star_w = 1.5  # true minimizer of the toy loss.
h_flat_w, h_steep_w = 1.0, 12.0  # two curvatures for the same optimum.
loss_flat_w = 0.5 * h_flat_w * (w_grid_w - w_star_w) ** 2  # shallow quadratic.
loss_steep_w = 0.5 * h_steep_w * (w_grid_w - w_star_w) ** 2  # steep quadratic.
print("flat gradient at w=3:", h_flat_w * (3.0 - w_star_w))
print("steep gradient at w=3:", h_steep_w * (3.0 - w_star_w))
assert h_steep_w * (3.0 - w_star_w) == 18.0

▶ What you'll see: the steep bowl produces a gradient of 18 at the same location where the flat bowl produces 1.5.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(w_grid_w, loss_flat_w, label="h=1 shallow")
plt.plot(w_grid_w, loss_steep_w, label="h=12 steep")
plt.axvline(w_star_w, color="black", linestyle="--", linewidth=1)
plt.scatter([3.0], [0.5 * h_steep_w * (3.0 - w_star_w) ** 2], color="red")
plt.title("1: same optimum, different curvature")
plt.xlabel("parameter w"); plt.ylabel("loss")
plt.legend(); plt.show()

▶ What you'll see: both losses minimize at 1.5, but the steep one rises much faster away from that point.

*Why it's done this way:* a gradient mixes two facts: direction to improve and local scale. The derivative \(h(w-w^*)\) is large either because we are far away or because the bowl is sharp. A second-order method separates those facts by dividing by curvature, so the update reflects distance-to-optimum rather than raw slope size.

### 2. Newton's step divides by curvature

For the same quadratic, the Hessian is just \(h\). Newton's update is \(w \leftarrow w - g/h\), so from any starting point it jumps exactly to \(w^*\) on a perfect quadratic. The lesson's scalar update \(2.000 - 0.050\cdot1.650=1.917\) is a first-order nudge; Newton asks whether the gradient should be shrunken or enlarged by the local curvature before nudging.

In [ ]:
w0_w = 3.0  # start to the right of the optimum.
g_w = h_steep_w * (w0_w - w_star_w)  # gradient in the steep bowl.
newton_step_w = g_w / h_steep_w  # curvature-corrected step length.
w_newton_w = w0_w - newton_step_w  # Newton update.
w_gd_w = w0_w - 0.05 * g_w  # ordinary gradient descent with a small learning rate.
print("gradient:", round(g_w, 3), "Newton step:", round(newton_step_w, 3))
print("GD w:", round(w_gd_w, 3), "Newton w:", round(w_newton_w, 3))
assert round(w_gd_w, 3) == 2.1
assert round(w_newton_w, 3) == 1.5

▶ What you'll see: gradient descent moves from 3.0 to 2.1, while Newton lands on 1.5 in one step.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(w_grid_w, loss_steep_w, color="purple")
plt.scatter([w0_w, w_gd_w, w_newton_w], [0.5*h_steep_w*(w0_w-w_star_w)**2,
                                         0.5*h_steep_w*(w_gd_w-w_star_w)**2,
                                         0.5*h_steep_w*(w_newton_w-w_star_w)**2],
            color=["red", "orange", "green"])
plt.title("2: curvature-corrected step")
plt.xlabel("w"); plt.ylabel("loss"); plt.show()

▶ What you'll see: the green Newton point is at the bottom, while the orange gradient-descent point is only partway there.

*Why it's done this way:* on a quadratic, the second-order Taylor model is exact: \(L(w+\Delta)\approx L(w)+g\Delta+\tfrac12 h\Delta^2\). Minimizing that approximation gives \(g+h\Delta=0\), hence \(\Delta=-g/h\). K-FAC keeps this same "divide by curvature" logic but replaces the impossible full Hessian with layerwise covariance factors.

### 3. A layer gradient is an outer product

For one linear layer with activations \(a\), pre-activations \(s=aW\), and backpropagated derivatives \(\delta=\partial L/\partial s\), the weight gradient is \(G=a^\top\delta\) for a single example and an average of outer products for a minibatch. This shape fact is why K-FAC can separate curvature into an activation factor and a gradient factor.

In [ ]:
a_w = np.array([[1.5, -0.5]])  # one input activation row, matching the lesson's scratch pass.
W_w = np.array([[1.0], [-0.1]])  # two weights into one unit.
b0_w = 0.7  # bias from the lesson content.
z_w = float(a_w @ W_w + b0_w)  # affine signal.
relu_w = max(0.0, z_w)  # gated signal.
print("affine signal:", round(z_w, 3), "ReLU signal:", round(relu_w, 3))
assert round(z_w, 3) == 2.25

▶ What you'll see: the two-input affine computation gives 2.25, and ReLU leaves it unchanged because it is positive.

In [ ]:
delta_w = np.array([[1.65]])  # derivative arriving from the loss for this one unit.
G_one_w = a_w.T @ delta_w  # outer product: input coordinate by output derivative.
print("single-example gradient:\n", np.round(G_one_w, 3))
assert np.allclose(G_one_w.ravel(), [2.475, -0.825])

▶ What you'll see: the first weight gets a positive gradient and the second gets a negative gradient.

In [ ]:
a_batch_w = np.array([[1.5, -0.5], [0.5, 1.0], [1.0, 0.0], [-1.0, 0.5]])
delta_batch_w = np.array([[1.65, -0.20], [0.50, 0.30], [1.00, -0.40], [-0.30, 0.20]])
G_batch_w = a_batch_w.T @ delta_batch_w / a_batch_w.shape[0]
print("batch gradient shape:", G_batch_w.shape)
print(np.round(G_batch_w, 3))
assert G_batch_w.shape == (2, 2)

▶ What you'll see: a 2×2 gradient matrix, one entry for each input-output weight connection.

*Why it's done this way:* backprop gives a local output-side error \(\delta\), but a weight only matters in proportion to the activation that flowed through it. Multiplying activation by derivative creates exactly the right local credit assignment. K-FAC's approximation starts from the observation that these two ingredients often have useful second-moment structure.

### 4. Curvature for a layer is enormous if we flatten every weight

If a layer has \(d_{in}\) inputs and \(d_{out}\) outputs, the flattened weight vector has \(d_{in}d_{out}\) entries. A full curvature matrix over those entries has \((d_{in}d_{out})^2\) numbers, and inverting it is the bottleneck. K-FAC avoids that by approximating the layer curvature as \(S\otimes A\), where \(A=E[a^\top a]\) and \(S=E[\delta^\top\delta]\).

In [ ]:
A_w = a_batch_w.T @ a_batch_w / a_batch_w.shape[0]  # activation covariance factor.
S_w = delta_batch_w.T @ delta_batch_w / delta_batch_w.shape[0]  # derivative covariance factor.
print("A factor:\n", np.round(A_w, 3))
print("S factor:\n", np.round(S_w, 3))
assert A_w.shape == (2, 2) and S_w.shape == (2, 2)

▶ What you'll see: two small 2×2 matrices summarizing input scale and output-gradient scale.

In [ ]:
F_kron_w = np.kron(S_w, A_w)  # full flattened curvature approximation for this tiny layer.
print("Kronecker curvature shape:", F_kron_w.shape)
print(np.round(F_kron_w, 3))
assert F_kron_w.shape == (4, 4)

▶ What you'll see: the two 2×2 factors expand into a 4×4 curvature matrix for the four flattened weights.

In [ ]:
din_w, dout_w = 1024, 1024
full_numbers_w = (din_w * dout_w) ** 2
factor_numbers_w = din_w ** 2 + dout_w ** 2
print("full curvature entries:", f"{full_numbers_w:,}")
print("K-FAC factor entries:", f"{factor_numbers_w:,}")
print("compression ratio:", int(full_numbers_w / factor_numbers_w))
assert int(full_numbers_w / factor_numbers_w) == 524288

▶ What you'll see: storing two factors is over 500,000× smaller than storing the full layer curvature at this size.

*Why it's done this way:* the identity \((S\otimes A)^{-1}=S^{-1}\otimes A^{-1}\) means we can act like we used a large curvature matrix while only inverting two small matrices. The approximation is a modeling choice: it assumes activation correlations and output-gradient correlations capture the most important curvature interactions separately.

### 5. The K-FAC update is two-sided preconditioning

K-FAC applies the inverse factors directly to the matrix-shaped gradient: \(\Delta W=-\eta A^{-1}GS^{-1}\). Left-multiplying by \(A^{-1}\) corrects input/activation scale; right-multiplying by \(S^{-1}\) corrects output-gradient scale. This is the concrete core formula from the lesson content.

In [ ]:
eta_w = 0.05  # small learning rate from the lesson content.
damp_w = 0.1  # damping keeps inverses stable.
A_damped_w = A_w + damp_w * np.eye(A_w.shape[0])
S_damped_w = S_w + damp_w * np.eye(S_w.shape[0])
A_inv_w = np.linalg.inv(A_damped_w)
S_inv_w = np.linalg.inv(S_damped_w)
Delta_kfac_w = -eta_w * A_inv_w @ G_batch_w @ S_inv_w
Delta_sgd_w = -eta_w * G_batch_w
print("SGD step:\n", np.round(Delta_sgd_w, 3))
print("K-FAC step:\n", np.round(Delta_kfac_w, 3))
assert Delta_kfac_w.shape == G_batch_w.shape

▶ What you'll see: the K-FAC step is not just a smaller copy of the gradient; it rotates and rescales entries by curvature.

In [ ]:
norm_sgd_w = float(np.linalg.norm(Delta_sgd_w))
norm_kfac_w = float(np.linalg.norm(Delta_kfac_w))
print("||SGD step||:", round(norm_sgd_w, 4))
print("||K-FAC step||:", round(norm_kfac_w, 4))
assert round(norm_sgd_w, 4) == 0.0519
plt.figure(figsize=(4.6, 3))
plt.bar(["SGD", "K-FAC"], [norm_sgd_w, norm_kfac_w], color=["gray", "seagreen"])
plt.title("5: curvature changes step size")
plt.ylabel("Frobenius norm of update"); plt.show()

▶ What you'll see: the two update norms differ because K-FAC measures distance in a curvature-aware geometry.

*Why it's done this way:* ordinary SGD treats every weight coordinate as equally scaled. But if one input feature has high variance or one output derivative is noisy, equal coordinate steps are not equal functional changes. The preconditioner whitens those two sides so the step is closer to a natural-gradient step in the layer's local coordinates.

### 6. Damping and minibatches keep the approximation usable

Covariance factors estimated from a minibatch can be singular or noisy. Damping adds \(\lambda I\) before inversion, making every eigenvalue at least \(\lambda\). That sacrifices some pure Newton aggressiveness for stability — the same practical theme as the lesson's scale, softmax, and memory bookkeeping.

In [ ]:
skinny_batch_w = np.array([[1.0, 2.0], [2.0, 4.0]])  # second column is exactly twice the first.
A_singular_w = skinny_batch_w.T @ skinny_batch_w / skinny_batch_w.shape[0]
eigs_raw_w = np.linalg.eigvalsh(A_singular_w)
eigs_damped_w = np.linalg.eigvalsh(A_singular_w + 0.1 * np.eye(2))
print("raw eigenvalues:", np.round(eigs_raw_w, 6))
print("damped eigenvalues:", np.round(eigs_damped_w, 6))
assert round(float(eigs_raw_w[0]), 6) == 0.0

▶ What you'll see: the raw factor has a zero eigenvalue, while damping lifts it to 0.1.

In [ ]:
cond_raw_w = np.inf if eigs_raw_w[0] == 0 else eigs_raw_w[-1] / eigs_raw_w[0]
cond_damped_w = eigs_damped_w[-1] / eigs_damped_w[0]
print("raw condition:", cond_raw_w)
print("damped condition:", round(float(cond_damped_w), 3))
assert round(float(cond_damped_w), 3) == 126.0
plt.figure(figsize=(4.6, 3))
plt.bar(["raw small eig", "damped small eig"], [eigs_raw_w[0], eigs_damped_w[0]], color=["crimson", "seagreen"])
plt.title("6: damping prevents divide-by-zero")
plt.ylabel("smallest eigenvalue"); plt.show()

▶ What you'll see: damping turns an impossible inverse into a stable, finite preconditioner.

*Why it's done this way:* preconditioning divides by estimated curvature. Dividing by zero or by a noisy tiny number would explode the update. Damping says "trust the curvature estimate, but not infinitely," which is why practical second-order methods look like careful engineering rather than magic.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, matrix products, covariance factors, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for compact diagnostic plots.
np.random.seed(0) # make the examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — One affine signal and ReLU gate

**Goal.** Reproduce the lesson's two-input scratch pass, because K-FAC still begins with ordinary forward signals before it approximates curvature.

In [ ]:
x_b1 = np.array([1.5, -0.5]) # define the two visible input activations.
w_b1 = np.array([1.0, -0.1]) # define the two weights into one hidden unit.
b_b1 = 0.7 # define the bias from the lesson arithmetic.
z_b1 = float(x_b1 @ w_b1 + b_b1) # compute 1.0*1.5 + -0.1*(-0.5) + 0.7.
h_b1 = max(0.0, z_b1) # apply ReLU gating.
print("affine:", round(z_b1, 3), "ReLU:", round(h_b1, 3))
assert round(z_b1, 3) == 2.25
plt.figure(figsize=(4, 3)); plt.bar(["x0w0", "x1w1", "bias", "ReLU"], [x_b1[0]*w_b1[0], x_b1[1]*w_b1[1], b_b1, h_b1], color="teal")
plt.title("Basic 1: signal pieces"); plt.xticks(rotation=20); plt.show()

▶ What you'll see: the two weighted inputs plus bias sum to 2.25, and ReLU passes the positive value through.

👀 Takeaway: K-FAC preconditions training, but the signals it summarizes are the same activations produced by the forward pass.

### Basic 2 — A scalar gradient step

**Goal.** Compute the lesson's small first-order update, because second-order methods modify this familiar gradient-descent baseline.

In [ ]:
theta_b2 = 2.0 # current scalar parameter.
eta_b2 = 0.05 # learning rate from the lesson content.
g_b2 = 1.65 # ordinary gradient.
theta_new_b2 = theta_b2 - eta_b2 * g_b2 # gradient descent update.
print("new theta:", round(theta_new_b2, 3))
assert round(theta_new_b2, 3) == 1.917
plt.figure(figsize=(4, 3)); plt.bar(["before", "after"], [theta_b2, theta_new_b2], color=["gray", "seagreen"])
plt.title("Basic 2: one gradient nudge"); plt.ylabel("parameter value"); plt.show()

▶ What you'll see: the parameter moves from 2.000 to 1.917, a deliberately modest change.

👀 Takeaway: first-order training changes parameters by repeated small steps proportional to the raw gradient.

### Basic 3 — Softmax turns scores into comparisons

**Goal.** Convert the lesson score 2.25 and baseline 0.40 into a probability, because losses and gradients usually depend on comparisons, not absolute scores.

In [ ]:
scores_b3 = np.array([2.25, 0.40]) # model score and baseline score.
exp_b3 = np.exp(scores_b3) # exponentiate scores into positive weights.
prob_b3 = exp_b3[0] / np.sum(exp_b3) # normalize the first score's weight.
print("exp values:", np.round(exp_b3, 3))
print("probability:", round(float(prob_b3), 3))
assert round(float(prob_b3), 3) == 0.864
plt.figure(figsize=(4, 3)); plt.bar(["model", "baseline"], exp_b3, color=["purple", "gray"])
plt.title("Basic 3: exponentiated evidence"); plt.show()

▶ What you'll see: exp(2.25) is much larger than exp(0.40), so the normalized probability is about 0.864.

👀 Takeaway: optimization often follows probabilities whose scale depends sharply on the raw scores.

### Basic 4 — Normalize one signal

**Goal.** Compute the lesson's normalized value, because curvature and gradients are meaningful only relative to signal scale.

In [ ]:
value_b4 = 2.25 # signal value from the forward pass.
mean_b4 = 1.0 # running or batch mean.
var_b4 = 0.25 # running or batch variance.
eps_b4 = 1e-5 # numerical stabilizer.
norm_b4 = (value_b4 - mean_b4) / np.sqrt(var_b4 + eps_b4) # standardized signal.
print("normalized value:", round(float(norm_b4), 3))
assert round(float(norm_b4), 3) == 2.5
plt.figure(figsize=(4, 3)); plt.bar(["raw", "mean", "normalized"], [value_b4, mean_b4, norm_b4], color="orange")
plt.title("Basic 4: scale bookkeeping"); plt.show()

▶ What you'll see: the signal is 2.5 standard deviations above the mean.

👀 Takeaway: scale bookkeeping changes how large an update effectively feels.

### Basic 5 — Count activation memory

**Goal.** Reproduce the lesson's activation memory calculation, because second-order methods are constrained by hardware as much as algebra.

In [ ]:
batch_b5 = 4 # number of activation vectors.
width_b5 = 128 # length of each activation vector.
bytes_per_float_b5 = 4 # 32-bit float storage.
kb_b5 = batch_b5 * width_b5 * bytes_per_float_b5 / 1024 # convert bytes to KB.
print("activation memory KB:", round(kb_b5, 3))
assert round(kb_b5, 3) == 2.0
plt.figure(figsize=(4, 3)); plt.bar(["activations"], [kb_b5], color="steelblue")
plt.title("Basic 5: tiny activation block memory"); plt.ylabel("KB"); plt.show()

▶ What you'll see: even a tiny 4×128 activation block uses 2 KB in float32.

👀 Takeaway: practical optimization methods must respect the memory footprint of saved activations and curvature factors.

### Basic 6 — Build an activation covariance factor

**Goal.** Compute \(A=E[a^\top a]\), because K-FAC's left factor summarizes activation scale and correlation.

In [ ]:
acts_b6 = np.array([[1.5, -0.5], [0.5, 1.0], [1.0, 0.0], [-1.0, 0.5]]) # four activation rows.
A_b6 = acts_b6.T @ acts_b6 / acts_b6.shape[0] # average outer product.
print("A:\n", np.round(A_b6, 3))
assert np.allclose(np.round(A_b6, 3), [[1.125, -0.188], [-0.188, 0.375]])
plt.figure(figsize=(4, 3)); plt.imshow(A_b6, cmap="viridis"); plt.colorbar(label="covariance")
plt.title("Basic 6: activation factor A"); plt.show()

▶ What you'll see: the first activation coordinate has larger variance than the second.

👀 Takeaway: K-FAC learns how input coordinates are scaled before choosing a weight update.

### Basic 7 — Build a gradient covariance factor

**Goal.** Compute \(S=E[\delta^\top\delta]\), because K-FAC's right factor summarizes output-gradient scale and correlation.

In [ ]:
deltas_b7 = np.array([[1.65, -0.20], [0.50, 0.30], [1.00, -0.40], [-0.30, 0.20]]) # backpropagated derivatives.
S_b7 = deltas_b7.T @ deltas_b7 / deltas_b7.shape[0] # average derivative outer product.
print("S:\n", np.round(S_b7, 3))
assert np.allclose(np.round(S_b7, 3), [[1.016, -0.16], [-0.16, 0.082]])
plt.figure(figsize=(4, 3)); plt.imshow(S_b7, cmap="magma"); plt.colorbar(label="covariance")
plt.title("Basic 7: gradient factor S"); plt.show()

▶ What you'll see: the first output derivative has much larger second moment than the second.

👀 Takeaway: output-side gradient scale is a separate part of the layer's curvature geometry.

### Basic 8 — Compute a batch weight gradient

**Goal.** Average activation-derivative outer products, because the ordinary gradient \(G\) is the object K-FAC preconditions.

In [ ]:
acts_b8 = np.array([[1.5, -0.5], [0.5, 1.0], [1.0, 0.0], [-1.0, 0.5]]) # activation batch.
deltas_b8 = np.array([[1.65, -0.20], [0.50, 0.30], [1.00, -0.40], [-0.30, 0.20]]) # derivative batch.
G_b8 = acts_b8.T @ deltas_b8 / acts_b8.shape[0] # matrix gradient for a 2x2 layer.
print("G:\n", np.round(G_b8, 3))
assert np.allclose(np.round(G_b8, 3), [[1.006, -0.188], [-0.119, 0.125]])
plt.figure(figsize=(4, 3)); plt.imshow(G_b8, cmap="coolwarm"); plt.colorbar(label="gradient")
plt.title("Basic 8: ordinary layer gradient"); plt.show()

▶ What you'll see: a 2×2 gradient with positive and negative entries.

👀 Takeaway: K-FAC does not replace gradients; it rescales the gradient matrix using curvature estimates.

### Basic 9 — Add damping before inversion

**Goal.** Stabilize a covariance inverse, because tiny eigenvalues would make a preconditioned step explode.

In [ ]:
A_b9 = np.array([[1.125, -0.1875], [-0.1875, 0.375]]) # activation factor.
damping_b9 = 0.1 # diagonal damping strength.
A_damped_b9 = A_b9 + damping_b9 * np.eye(2) # lift every eigenvalue by damping.
eigs_b9 = np.linalg.eigvalsh(A_damped_b9) # inspect numerical stability.
print("damped eigenvalues:", np.round(eigs_b9, 3))
assert np.all(eigs_b9 > 0.1)
plt.figure(figsize=(4, 3)); plt.bar(["eig0", "eig1"], eigs_b9, color="seagreen")
plt.title("Basic 9: damped curvature eigenvalues"); plt.show()

▶ What you'll see: both damped eigenvalues are safely positive.

👀 Takeaway: damping limits how aggressively the inverse can amplify uncertain directions.

### Basic 10 — Apply the K-FAC formula once

**Goal.** Compute \(-\eta A^{-1}GS^{-1}\), because this is the lesson's central curvature-preconditioned update.

In [ ]:
A_b10 = np.array([[1.125, -0.1875], [-0.1875, 0.375]]) # activation factor.
S_b10 = np.array([[1.015625, -0.16], [-0.16, 0.0825]]) # gradient factor.
G_b10 = np.array([[1.00625, -0.1875], [-0.11875, 0.125]]) # ordinary gradient.
eta_b10 = 0.05 # learning rate.
damp_b10 = 0.1 # damping for both factors.
Delta_b10 = -eta_b10 * np.linalg.inv(A_b10 + damp_b10*np.eye(2)) @ G_b10 @ np.linalg.inv(S_b10 + damp_b10*np.eye(2))
print("K-FAC update:\n", np.round(Delta_b10, 3))
assert Delta_b10.shape == (2, 2)
plt.figure(figsize=(4, 3)); plt.imshow(Delta_b10, cmap="coolwarm"); plt.colorbar(label="update")
plt.title("Basic 10: preconditioned step"); plt.show()

▶ What you'll see: the update is a curvature-shaped matrix rather than a uniform multiple of `G_b10`.

👀 Takeaway: K-FAC's practical step is two small inverses wrapped around the ordinary gradient.

## 🟡 Easy

### Easy 1 — Compare SGD and K-FAC directions

**Goal.** Put first-order and K-FAC updates side by side, because preconditioning changes both magnitude and direction.

In [ ]:
A_e1 = np.array([[1.125, -0.1875], [-0.1875, 0.375]]) # activation covariance.
S_e1 = np.array([[1.015625, -0.16], [-0.16, 0.0825]]) # derivative covariance.
G_e1 = np.array([[1.00625, -0.1875], [-0.11875, 0.125]]) # ordinary gradient.
eta_e1, damp_e1 = 0.05, 0.1 # learning rate and damping.
sgd_e1 = -eta_e1 * G_e1 # first-order update.
kfac_e1 = -eta_e1 * np.linalg.inv(A_e1 + damp_e1*np.eye(2)) @ G_e1 @ np.linalg.inv(S_e1 + damp_e1*np.eye(2))
cos_e1 = float(np.sum(sgd_e1 * kfac_e1) / (np.linalg.norm(sgd_e1) * np.linalg.norm(kfac_e1)))
print("||SGD||:", round(float(np.linalg.norm(sgd_e1)), 4), "||K-FAC||:", round(float(np.linalg.norm(kfac_e1)), 4))
print("direction cosine:", round(cos_e1, 3))
assert round(float(np.linalg.norm(sgd_e1)), 4) == 0.0519
plt.figure(figsize=(4, 3)); plt.bar(["SGD", "K-FAC"], [np.linalg.norm(sgd_e1), np.linalg.norm(kfac_e1)], color=["gray", "green"])
plt.title("Easy 1: update norms"); plt.show()

▶ What you'll see: K-FAC is not merely `SGD` with a different learning rate; the direction cosine is below 1.

👀 Takeaway: preconditioning changes the geometry of the step, not just its scalar size.

### Easy 2 — Verify the Kronecker inverse identity

**Goal.** Check \((S\otimes A)^{-1}=S^{-1}\otimes A^{-1}\), because this identity is what makes K-FAC computationally attractive.

In [ ]:
A_e2 = np.array([[1.2, 0.2], [0.2, 0.7]]) # positive definite activation factor.
S_e2 = np.array([[0.9, -0.1], [-0.1, 0.4]]) # positive definite derivative factor.
left_e2 = np.linalg.inv(np.kron(S_e2, A_e2)) # inverse of the full Kronecker matrix.
right_e2 = np.kron(np.linalg.inv(S_e2), np.linalg.inv(A_e2)) # Kronecker of small inverses.
err_e2 = float(np.max(np.abs(left_e2 - right_e2))) # maximum numerical difference.
print("max identity error:", err_e2)
assert err_e2 < 1e-12
plt.figure(figsize=(4, 3)); plt.imshow(left_e2 - right_e2, cmap="coolwarm"); plt.colorbar(label="difference")
plt.title("Easy 2: inverse identity error"); plt.show()

▶ What you'll see: the difference heatmap is essentially zero everywhere.

👀 Takeaway: two small inverses can represent the action of one much larger structured inverse.

### Easy 3 — Show damping shrinks an aggressive step

**Goal.** Sweep damping values, because larger damping moves the method away from pure Newton and toward safer, smaller updates.

In [ ]:
A_e3 = np.array([[1.125, -0.1875], [-0.1875, 0.375]])
S_e3 = np.array([[1.015625, -0.16], [-0.16, 0.0825]])
G_e3 = np.array([[1.00625, -0.1875], [-0.11875, 0.125]])
damps_e3 = np.array([0.01, 0.05, 0.1, 0.5, 1.0]) # stability strengths to compare.
norms_e3 = []
for d_e3 in damps_e3:
    step_e3 = -0.05 * np.linalg.inv(A_e3 + d_e3*np.eye(2)) @ G_e3 @ np.linalg.inv(S_e3 + d_e3*np.eye(2))
    norms_e3.append(float(np.linalg.norm(step_e3)))
print("step norms:", np.round(norms_e3, 4))
assert norms_e3[0] > norms_e3[-1]
plt.figure(figsize=(4.6, 3)); plt.plot(damps_e3, norms_e3, marker="o", color="crimson")
plt.title("Easy 3: damping vs update size"); plt.xlabel("damping"); plt.ylabel("step norm"); plt.show()

▶ What you'll see: the update norm falls as damping increases.

👀 Takeaway: damping is a safety knob that prevents noisy curvature estimates from taking oversized steps.

### Easy 4 — Estimate factors from two minibatches

**Goal.** Compare covariance estimates across minibatches, because K-FAC factors are statistical estimates rather than exact constants.

In [ ]:
rng_e4 = np.random.default_rng(4) # reproducible minibatches.
acts1_e4 = rng_e4.normal(loc=0.0, scale=1.0, size=(8, 2)) # first activation minibatch.
acts2_e4 = rng_e4.normal(loc=0.0, scale=1.0, size=(8, 2)) # second activation minibatch.
A1_e4 = acts1_e4.T @ acts1_e4 / acts1_e4.shape[0] # first factor estimate.
A2_e4 = acts2_e4.T @ acts2_e4 / acts2_e4.shape[0] # second factor estimate.
diff_e4 = float(np.linalg.norm(A1_e4 - A2_e4))
print("A1:\n", np.round(A1_e4, 3))
print("A2:\n", np.round(A2_e4, 3))
print("factor difference norm:", round(diff_e4, 3))
assert diff_e4 > 0.1
plt.figure(figsize=(4, 3)); plt.bar(["batch1 trace", "batch2 trace"], [np.trace(A1_e4), np.trace(A2_e4)], color="purple")
plt.title("Easy 4: minibatch factor variability"); plt.show()

▶ What you'll see: the two minibatches produce similar-shaped but not identical covariance factors.

👀 Takeaway: K-FAC must balance curvature information with estimator noise from minibatches.

### Easy 5 — Translate factor storage into memory savings

**Goal.** Count entries for full curvature versus K-FAC factors, because the approximation exists to make second-order information fit in memory.

In [ ]:
sizes_e5 = np.array([16, 64, 256, 1024]) # square layer widths to compare.
full_entries_e5 = (sizes_e5 * sizes_e5) ** 2 # full curvature entries for W flattened.
factor_entries_e5 = sizes_e5 ** 2 + sizes_e5 ** 2 # A plus S entries.
ratios_e5 = full_entries_e5 / factor_entries_e5
print("ratios:", ratios_e5.astype(int))
assert int(ratios_e5[-1]) == 524288
plt.figure(figsize=(5, 3)); plt.plot(sizes_e5, ratios_e5, marker="o", color="teal")
plt.title("Easy 5: K-FAC storage ratio"); plt.xlabel("layer width"); plt.ylabel("full entries / factor entries"); plt.yscale("log"); plt.show()

▶ What you'll see: the storage advantage grows rapidly with layer width.

👀 Takeaway: K-FAC keeps second-order training practical by replacing one giant matrix with two layer-sized factors.

## 🔴 Advanced

### Advanced 1 — Match matrix preconditioning to full Kronecker preconditioning

**Goal.** Verify that the matrix formula \(A^{-1}GS^{-1}\) matches applying \((S\otimes A)^{-1}\) to a flattened gradient when column-major vectorization is used.

In [ ]:
A_a1 = np.array([[1.2, 0.2], [0.2, 0.7]])
S_a1 = np.array([[0.9, -0.1], [-0.1, 0.4]])
G_a1 = np.array([[1.0, -0.3], [0.2, 0.5]])
mat_step_a1 = np.linalg.inv(A_a1) @ G_a1 @ np.linalg.inv(S_a1) # two-sided K-FAC action.
vecG_a1 = G_a1.reshape(-1, order="F") # column-major vectorization.
full_step_a1 = np.linalg.inv(np.kron(S_a1, A_a1)) @ vecG_a1 # full Kronecker inverse action.
max_err_a1 = float(np.max(np.abs(mat_step_a1.reshape(-1, order="F") - full_step_a1)))
print("max equivalence error:", max_err_a1)
assert max_err_a1 < 1e-12
plt.figure(figsize=(4, 3)); plt.imshow(mat_step_a1, cmap="coolwarm"); plt.colorbar(label="preconditioned gradient")
plt.title("Advanced 1: matrix-form preconditioner"); plt.show()

▶ What you'll see: the numerical error is near machine precision, confirming both views are the same operation.

👀 Takeaway: K-FAC is not hand-wavy matrix decoration; it is the structured full-curvature operation written efficiently.

### Advanced 2 — Train a tiny linear model with SGD versus K-FAC

**Goal.** Fit a noisy linear regression with both updates, because curvature-aware scaling should reduce loss quickly when factor estimates are stable.

In [ ]:
rng_a2 = np.random.default_rng(2)
X_a2 = rng_a2.normal(size=(40, 2)) # design matrix / activations.
true_W_a2 = np.array([[1.5], [-2.0]]) # target linear weights.
y_a2 = X_a2 @ true_W_a2 + 0.05 * rng_a2.normal(size=(40, 1)) # noisy targets.
W_sgd_a2 = np.zeros((2, 1)); W_kfac_a2 = np.zeros((2, 1)) # two models from same start.
loss_sgd_a2 = []; loss_kfac_a2 = []
for epoch_a2 in range(30):
    err_sgd_a2 = X_a2 @ W_sgd_a2 - y_a2
    G_sgd_a2 = X_a2.T @ err_sgd_a2 / X_a2.shape[0]
    W_sgd_a2 -= 0.2 * G_sgd_a2
    err_kfac_a2 = X_a2 @ W_kfac_a2 - y_a2
    G_kfac_a2 = X_a2.T @ err_kfac_a2 / X_a2.shape[0]
    A_kfac_a2 = X_a2.T @ X_a2 / X_a2.shape[0] + 0.05 * np.eye(2)
    W_kfac_a2 -= 0.8 * np.linalg.inv(A_kfac_a2) @ G_kfac_a2
    loss_sgd_a2.append(float(np.mean((X_a2 @ W_sgd_a2 - y_a2) ** 2)))
    loss_kfac_a2.append(float(np.mean((X_a2 @ W_kfac_a2 - y_a2) ** 2)))
print("final losses SGD/K-FAC:", round(loss_sgd_a2[-1], 5), round(loss_kfac_a2[-1], 5))
assert loss_kfac_a2[-1] < loss_sgd_a2[-1]
plt.figure(figsize=(5, 3)); plt.plot(loss_sgd_a2, label="SGD"); plt.plot(loss_kfac_a2, label="K-FAC-like")
plt.title("Advanced 2: curvature-aware convergence"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.show()

▶ What you'll see: the K-FAC-like curve drops faster and ends below the SGD curve for this scaled linear problem.

👀 Takeaway: when the curvature approximation is accurate, preconditioning can convert slow zig-zagging into direct progress.

### Advanced 3 — Show why scale imbalance hurts SGD

**Goal.** Rescale one input feature by 20×, because first-order updates become ill-conditioned when coordinates have very different curvature.

In [ ]:
rng_a3 = np.random.default_rng(3)
X_a3 = rng_a3.normal(size=(60, 2)); X_a3[:, 0] *= 20.0 # one high-scale feature.
true_W_a3 = np.array([[0.1], [2.0]])
y_a3 = X_a3 @ true_W_a3
A_a3 = X_a3.T @ X_a3 / X_a3.shape[0]
eigs_a3 = np.linalg.eigvalsh(A_a3)
condition_a3 = float(eigs_a3[-1] / eigs_a3[0])
print("feature curvature eigenvalues:", np.round(eigs_a3, 3))
print("condition number:", round(condition_a3, 1))
assert condition_a3 > 100
plt.figure(figsize=(4, 3)); plt.bar(["small eig", "large eig"], eigs_a3, color=["orange", "crimson"])
plt.title("Advanced 3: scale imbalance curvature"); plt.yscale("log"); plt.show()

▶ What you'll see: the large-scale feature creates a much larger curvature eigenvalue.

👀 Takeaway: K-FAC's activation factor directly targets the scale imbalance that forces SGD to use tiny steps.

### Advanced 4 — Track eigenvalues before and after damping

**Goal.** Inspect eigenvalues across a range of damping values, because the inverse preconditioner is only safe when its smallest eigenvalues are controlled.

In [ ]:
A_a4 = np.array([[2.0, 1.99], [1.99, 1.9801]]) # nearly rank-one covariance.
damps_a4 = np.array([0.0, 1e-3, 1e-2, 1e-1, 1.0])
smallest_a4 = []
conds_a4 = []
for d_a4 in damps_a4:
    eigs_a4 = np.linalg.eigvalsh(A_a4 + d_a4 * np.eye(2))
    smallest_a4.append(float(eigs_a4[0]))
    conds_a4.append(float(eigs_a4[-1] / eigs_a4[0]))
print("smallest eigenvalues:", np.round(smallest_a4, 6))
print("conditions:", np.round(conds_a4, 1))
assert conds_a4[-1] < conds_a4[1]
plt.figure(figsize=(5, 3)); plt.plot(damps_a4, conds_a4, marker="o", color="navy")
plt.title("Advanced 4: damping improves conditioning"); plt.xlabel("damping"); plt.ylabel("condition number"); plt.yscale("log"); plt.show()

▶ What you'll see: as damping grows, the condition number falls by orders of magnitude.

👀 Takeaway: damping is the practical bridge between a mathematically attractive inverse and a numerically safe update.

### Advanced 5 — Compare full curvature memory with factor memory in MB

**Goal.** Convert entry counts into megabytes, because the full second-order matrix becomes impossible before the algebra becomes confusing.

In [ ]:
widths_a5 = np.array([64, 128, 256, 512]) # square layer widths.
float_bytes_a5 = 4 # float32 storage.
full_mb_a5 = ((widths_a5 * widths_a5) ** 2) * float_bytes_a5 / (1024 ** 2) # full curvature memory.
factor_mb_a5 = (2 * widths_a5 ** 2) * float_bytes_a5 / (1024 ** 2) # A and S memory.
print("full MB:", np.round(full_mb_a5, 1))
print("factor MB:", np.round(factor_mb_a5, 3))
assert round(float(full_mb_a5[-1]), 1) == 262144.0
plt.figure(figsize=(5, 3)); plt.plot(widths_a5, full_mb_a5, marker="o", label="full curvature")
plt.plot(widths_a5, factor_mb_a5, marker="o", label="K-FAC factors")
plt.yscale("log"); plt.title("Advanced 5: memory wall"); plt.xlabel("layer width"); plt.ylabel("MB, log scale"); plt.legend(); plt.show()

▶ What you'll see: at width 512, full curvature is hundreds of gigabytes while factors are only a few megabytes.

👀 Takeaway: K-FAC is a second-order compromise designed for the memory scale of neural-network layers.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

K-FAC approximates curvature with Kronecker factors so a neural-network second-order step becomes practical.

Full curvature is too large for neural networks. K-FAC keeps a real matrix preconditioner by approximating each layer's curvature with activation and gradient covariance factors. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def split_scale(X, y):
    if len(y) > 300:
        x_small, _, y_small, _ = train_test_split(X, y, train_size=300, random_state=6, stratify=y)
    else:
        x_small = X
        y_small = y
    x_tr, x_te, y_tr, y_te = train_test_split(x_small, y_small, test_size=0.4, random_state=0, stratify=y_small)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=1, keepdims=True)


def relu(z):
    return np.maximum(z, 0.0)


def init_weights(n_in, n_hidden, n_out, mode, seed):
    rng = np.random.default_rng(seed)
    if mode == "xavier":
        scale1 = math.sqrt(2.0 / (n_in + n_hidden))
        scale2 = math.sqrt(2.0 / (n_hidden + n_out))
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "he":
        scale1 = math.sqrt(2.0 / n_in)
        scale2 = math.sqrt(2.0 / n_hidden)
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "tiny":
        W1 = rng.normal(0.0, 0.01, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.01, size=(n_hidden, n_out))
    elif mode == "large":
        W1 = rng.normal(0.0, 2.0, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 2.0, size=(n_hidden, n_out))
    elif mode == "orthogonal":
        Q1, _ = np.linalg.qr(rng.normal(size=(n_in, max(n_in, n_hidden))))
        Q2, _ = np.linalg.qr(rng.normal(size=(n_hidden, max(n_hidden, n_out))))
        W1 = Q1[:, :n_hidden]
        W2 = Q2[:, :n_out]
    else:
        W1 = rng.normal(0.0, 0.1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.1, size=(n_hidden, n_out))
    b1 = np.zeros(n_hidden)
    b2 = np.zeros(n_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(params, X, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    W1 = params["W1"]
    if dropconnect_p > 0.0 and rng is not None:
        keep_w = 1.0 - dropconnect_p
        mask_w = rng.binomial(1, keep_w, size=W1.shape) / keep_w
        W1 = W1 * mask_w
    z1 = X @ W1 + params["b1"]
    h1 = relu(z1)
    mask = None
    if dropout_p > 0.0 and rng is not None:
        keep = 1.0 - dropout_p
        mask = rng.binomial(1, keep, size=h1.shape) / keep
        h1 = h1 * mask
    logits = h1 @ params["W2"] + params["b2"]
    return z1, h1, logits, mask


def loss_and_grads(params, X, y, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    classes = params["b2"].shape[0]
    z1, h1, logits, mask = forward(params, X, dropout_p, rng, dropconnect_p)
    probs = softmax(logits)
    target = one_hot(y, classes)
    loss = -np.mean(np.sum(target * np.log(probs + 1e-12), axis=1))
    dlogits = (probs - target) / len(y)
    dW2 = h1.T @ dlogits
    db2 = np.sum(dlogits, axis=0)
    dh1 = dlogits @ params["W2"].T
    if mask is not None:
        dh1 = dh1 * mask
    dz1 = dh1 * (z1 > 0.0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads


def predict(params, X):
    _, _, logits, _ = forward(params, X)
    return np.argmax(logits, axis=1)


def eval_loss(params, X, y):
    classes = params["b2"].shape[0]
    _, _, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    return -float(np.mean(np.sum(target * np.log(probs + 1e-12), axis=1)))


def vector_norm(params):
    total = 0.0
    for value in params.values():
        total += float(np.sum(value * value))
    return math.sqrt(total)


def kfac_precondition_grads(params, X, y, grads, damping):
    classes = params["b2"].shape[0]
    z1, h1, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    dlogits = (probs - target) / len(y)
    dh1 = dlogits @ params["W2"].T
    dz1 = dh1 * (z1 > 0.0)
    out = {key: value.copy() for key, value in grads.items()}
    A1 = X.T @ X / len(y) + damping * np.eye(X.shape[1])
    S1 = dz1.T @ dz1 / len(y) + damping * np.eye(dz1.shape[1])
    A2 = h1.T @ h1 / len(y) + damping * np.eye(h1.shape[1])
    S2 = dlogits.T @ dlogits / len(y) + damping * np.eye(dlogits.shape[1])
    out["W1"] = np.linalg.solve(A1, grads["W1"]) @ np.linalg.inv(S1)
    out["W2"] = np.linalg.solve(A2, grads["W2"]) @ np.linalg.inv(S2)
    return out


def apply_update(params, grads, state, method, lr, t, config):
    beta1 = config.get("beta1", 0.9)
    beta2 = config.get("beta2", 0.999)
    eps = config.get("eps", 1e-8)
    mu = config.get("momentum", 0.0)
    weight_decay = config.get("weight_decay", 0.0)
    for key in params:
        grad = grads[key]
        if method == "sgd":
            update = -lr * grad
        elif method == "momentum":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        elif method == "adagrad":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc += grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "rmsprop":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc *= beta2
            acc += (1.0 - beta2) * grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "adam" or method == "adamw":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            update = -lr * m_hat / (np.sqrt(v_hat) + eps)
            if method == "adamw" and key.startswith("W"):
                update -= lr * weight_decay * params[key]
        elif method == "lion":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            blended = beta1 * m + (1.0 - beta1) * grad
            update = -lr * np.sign(blended)
            m *= beta2
            m += (1.0 - beta2) * grad
        elif method == "lamb":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            raw = m / (np.sqrt(v) + eps)
            if key.startswith("W"):
                raw += weight_decay * params[key]
            ratio = np.linalg.norm(params[key]) / (np.linalg.norm(raw) + eps)
            ratio = float(np.clip(ratio, 0.1, 10.0))
            update = -lr * ratio * raw
        elif method == "nesterov":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        else:
            update = -lr * grad
        if weight_decay > 0.0 and method not in ["adamw", "lamb"] and key.startswith("W"):
            update -= lr * weight_decay * params[key]
        params[key] += update


def train_mlp(x_tr, y_tr, x_te, y_te, method="sgd", init="he", epochs=12, lr=0.05, hidden=8, batch_size=None, config=None, dropout_p=0.0, dropconnect_p=0.0, seed=0, early_patience=None):
    if config is None:
        config = {}
    classes = int(np.max(y_tr)) + 1
    params = init_weights(x_tr.shape[1], hidden, classes, init, seed)
    state = {}
    rng = np.random.default_rng(seed + 100)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "norm": []}
    best_loss = float("inf")
    best_params = None
    bad_epochs = 0
    n = len(y_tr)
    if batch_size is None:
        batch_size = n
    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            if method == "nesterov":
                lookahead = {}
                for key in params:
                    velocity = state.setdefault("v_" + key, np.zeros_like(params[key]))
                    lookahead[key] = params[key].copy()
                    params[key] += config.get("momentum", 0.9) * velocity
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
                for key in params:
                    params[key] = lookahead[key]
            else:
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
            if method == "kfac":
                grads = kfac_precondition_grads(params, x_tr[idx], y_tr[idx], grads, config.get("damping", 0.03))
                apply_update(params, grads, state, "sgd", lr, epoch, config)
            else:
                apply_update(params, grads, state, method, lr, epoch, config)
        train_loss = eval_loss(params, x_tr, y_tr)
        val_loss = eval_loss(params, x_te, y_te)
        train_acc = accuracy_score(y_tr, predict(params, x_tr))
        val_acc = accuracy_score(y_te, predict(params, x_te))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["norm"].append(vector_norm(params))
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = {key: value.copy() for key, value in params.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if early_patience is not None and bad_epochs >= early_patience:
            params = best_params
            break
    return params, history


def run_component_ladder(variants, metric="accuracy", epochs=12, hidden=16):
    rows = []
    histories = {}
    artifacts = {}
    for rung_index, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        histories[name] = {}
        artifacts[name] = {}
        for variant in variants:
            params, hist = train_mlp(
                x_tr,
                y_tr,
                x_te,
                y_te,
                method=variant.get("method", "sgd"),
                init=variant.get("init", "he"),
                epochs=variant.get("epochs", epochs),
                lr=variant.get("lr", 0.05),
                hidden=hidden,
                batch_size=variant.get("batch_size"),
                config=variant.get("config", {}),
                dropout_p=variant.get("dropout_p", 0.0),
                dropconnect_p=variant.get("dropconnect_p", 0.0),
                seed=variant.get("seed", 10 + rung_index),
                early_patience=variant.get("early_patience"),
            )
            preds = predict(params, x_te)
            acc = accuracy_score(y_te, preds)
            val_loss = eval_loss(params, x_te, y_te)
            value = acc if metric == "accuracy" else val_loss
            rows.append({"rung": name, "variant": variant["name"], "accuracy": acc, "loss": val_loss, "metric": value})
            histories[name][variant["name"]] = hist
            artifacts[name][variant["name"]] = (x_te, y_te, preds)
    return rows, histories, artifacts


def print_table(rows, metric_name):
    print(f"{'rung':34s} {'variant':18s} {metric_name:>10s} {'acc':>8s} {'loss':>8s}")
    for row in rows:
        print(f"{row['rung'][:34]:34s} {row['variant'][:18]:18s} {row['metric']:10.3f} {row['accuracy']:8.3f} {row['loss']:8.3f}")


def plot_results(rows, histories, artifacts, metric_name, best_variant):
    rung_names = list(histories.keys())
    fig, axes = plt.subplots(2, len(rung_names), figsize=(3.2 * len(rung_names), 6.4))
    for col, rung in enumerate(rung_names):
        x_te, y_te, preds = artifacts[rung][best_variant]
        if x_te.shape[1] > 2:
            shown = PCA(n_components=2, random_state=0).fit_transform(x_te)
        else:
            shown = x_te[:, :2]
        axes[0, col].scatter(shown[:, 0], shown[:, 1], c=preds, s=12, cmap="tab10", alpha=0.85)
        axes[0, col].set_title(rung.split("(")[0].strip())
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])
        hist = histories[rung][best_variant]
        curve_key = "val_acc" if metric_name == "accuracy" else "val_loss"
        axes[1, col].plot(hist[curve_key], label=best_variant)
        axes[1, col].set_xlabel("epoch")
        axes[1, col].set_title(metric_name)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    variants = sorted({row["variant"] for row in rows})
    for variant in variants:
        vals = [row["metric"] for row in rows if row["variant"] == variant]
        ax.plot(range(1, len(vals) + 1), vals, marker="o", label=variant)
    ax.set_xticks(range(1, len(rung_names) + 1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(rung_names) + 1)])
    ax.set_ylabel(metric_name)
    ax.set_title("Same ladder, component varied")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## The concept, built once: Kronecker preconditioning

The lesson formula is
$$\Delta W\approx -\eta\,A^{-1}G\,S^{-1}.$$
The lesson's scalar training knob has $\eta=0.050$, $g=1.650$, and $\theta=2.000$, so the plain move is $1.9175$.

In [ ]:

def preconditioned_step(A, G, S, eta, damping):
    left = np.linalg.inv(A + damping * np.eye(A.shape[0]))
    right = np.linalg.inv(S + damping * np.eye(S.shape[0]))
    return -eta * left @ G @ right

plain_theta = 2.0 - 0.050 * 1.650
A = np.array([[2.0, 0.5], [0.5, 1.0]])
G = np.array([[0.3, -0.1], [0.2, 0.4]])
S = np.array([[1.5, 0.2], [0.2, 1.2]])
delta = preconditioned_step(A, G, S, 0.05, 0.01)
print(plain_theta)
print(np.round(delta, 4))
assert abs(plain_theta - 1.9175) < 1e-12
assert delta.shape == (2, 2)


The matrix inverse is real, but it is local to a block. In the training sweep below, a diagonal K-FAC-like preconditioner keeps a moving curvature estimate for every parameter block.

In [ ]:

full_curvature_entries = 64 * 16 * 64 * 16
block_entries = 64 * 64 + 16 * 16
print("full entries", full_curvature_entries)
print("K-FAC factor entries", block_entries)
assert full_curvature_entries == 1048576
assert block_entries == 4352


## The dataset ladder

Every topic uses the same `clf_digits_ladder()` and the same small MLP. Only the named optimizer or regularization component changes from variant to variant.

In [ ]:

rungs = clf_digits_ladder()
for name, X, y in rungs:
    classes = np.unique(y)
    print(f"{name:38s} shape={X.shape} classes={len(classes)} sample_y={y[:8].tolist()}")
print("D1 sample X:")
print(rungs[0][1])


## Run the same method across D1-D5

The architecture, splits, scaling, and seed policy stay fixed. The table reports one comparable metric per rung.

In [ ]:

variants = [
    {"name": "first-order", "method": "sgd", "lr": 0.05},
    {"name": "momentum", "method": "momentum", "lr": 0.04, "config": {"momentum": 0.9}},
    {"name": "K-FAC", "method": "kfac", "lr": 0.002, "config": {"damping": 0.05}},
]

rows, histories, artifacts = run_component_ladder(variants, metric="loss", epochs=10, hidden=8)
print_table(rows, "loss")


## Results visualization

Top row: small multiples of held-out predictions. Bottom row: validation curves for the highlighted variant, followed by the component summary curve.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:

    plot_results(rows, histories, artifacts, "loss", "KFAC-diag")

except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Pitfall on D5: curvature shape and memory

For D5, storing full curvature is wasteful. The fix is exactly the K-FAC idea: store factors or diagonal/block approximations instead of the full matrix.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
n_in = X.shape[1]
hidden = 16
full_bytes = (n_in * hidden) ** 2 * 8
factor_bytes = (n_in * n_in + hidden * hidden) * 8
diag_bytes = (n_in * hidden) * 8
print("full curvature MB", round(full_bytes / 1_000_000, 3))
print("K-FAC factor MB", round(factor_bytes / 1_000_000, 3))
print("diagonal MB", round(diag_bytes / 1_000_000, 3))
assert factor_bytes < full_bytes
assert diag_bytes < factor_bytes


## Evaluate it + Practice

- Main metric: held-out loss on every D1-D5 rung, compared with a no-skill baseline near random guessing.
- Sanity check: D1 XOR should improve above chance once the hidden ReLU layer is active.
- Ablation: replace the preconditioned update with first-order SGD; the metric should drop or the curve should become less stable.
- Failure signals: exploding loss, flat accuracy near chance, or a D5 train/validation gap that moves in opposite directions.
- Reproducibility: seeds are fixed and the ladder uses sklearn-bundled data only.

Practice 1: Change one hyperparameter in the strongest variant and rerun the summary curve.

Practice 2: Add a new diagnostic printout that distinguishes train accuracy from validation accuracy.

Practice 3: Explain why D5 is harder than D1 using the table and one plotted curve.